# 02b — Depuración Silver genérica (datos producidos por el agente)

Versión de `02_depuracion_silver.ipynb` que **no asume ningún hecho del
restaurante original** (ni "cierra los lunes", ni el mapeo exacto de
`status`, ni la lista de eventos a excluir...). Usa las funciones genéricas
de `tfm_depuracion.py` y unas pocas comprobaciones estadísticas propias,
pensadas para funcionar igual con los datos de cualquier restaurante que
haya pasado por `tfm_agente2.py`.

**Regla de diseño, para que sepas qué esperar:**
- Todo lo que es una comprobación estadística que solo AÑADE una columna de
  aviso (`flag_...`) se aplica **automáticamente**, sin preguntar — es
  reversible, no borra ni modifica nada.
- Todo lo que **borra filas/columnas, modifica valores o fusiona
  categorías** se te pregunta por consola (con un valor por defecto
  sensato) antes de aplicarse. Si respondes que no, los datos se quedan
  como estaban (como mucho con el flag informativo).

No sustituye al criterio humano — allí donde la decisión depende de
conocer el negocio (¿qué categorías de evento importan? ¿qué días cierra
este restaurante?), te lo pregunta en vez de inventarlo.

## Sección 0 — Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

# ── Rutas del proyecto ────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Cambia esto si el Bronze del agente está en otra carpeta.
BRONZE_SNAP = PROJECT_ROOT / 'agente' / 'prueba3_bronze'
SILVER_SNAP = PROJECT_ROOT / 'agente' / 'prueba3_silver'
SILVER_SNAP.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Bronze (agente): {BRONZE_SNAP}')
print(f'Silver (agente): {SILVER_SNAP}')

Project root : /Users/laura/TFM-Hosteleria-AI
Bronze (agente): /Users/laura/TFM-Hosteleria-AI/agente/prueba3_bronze
Silver (agente): /Users/laura/TFM-Hosteleria-AI/agente/prueba3_silver


### 0.1 Carga de parquets Bronze\n\nSe cargan solo las entidades que el agente haya llegado a producir — si falta alguna, se avisa y se continúa sin ella, en vez de fallar todo el notebook.

In [2]:
NOMBRES_ENTIDAD = [
    'tickets', 'ventas', 'reservas', 'tips', 'articulos', 'departamentos',
    'menu', 'festivos', 'eventos', 'meteo_diaria', 'meteo_horaria',
    'total_articles', 'facturas',
]

datasets = {}
faltantes = []
for nombre in NOMBRES_ENTIDAD:
    ruta = BRONZE_SNAP / f'{nombre}_raw.parquet'
    if ruta.exists():
        datasets[nombre] = pd.read_parquet(ruta)
    else:
        faltantes.append(nombre)

print(f'{len(datasets)}/{len(NOMBRES_ENTIDAD)} datasets cargados.')
if faltantes:
    print(f'Faltan (el agente no las generó, o no se aprobaron): {faltantes}')
    print('Las secciones correspondientes de este notebook se saltarán automáticamente.')

6/13 datasets cargados.
Faltan (el agente no las generó, o no se aprobaron): ['tips', 'departamentos', 'menu', 'festivos', 'eventos', 'meteo_diaria', 'meteo_horaria']
Las secciones correspondientes de este notebook se saltarán automáticamente.


## Sección 1 — Auditoría inicial

In [3]:
def audit_dataset(nombre, df, col_fecha=None):
    """Genera un resumen de calidad de un DataFrame. (Igual que en 02_original.)"""
    fecha_min = pd.NaT
    fecha_max = pd.NaT
    if col_fecha is not None and col_fecha in df.columns:
        col = pd.to_datetime(df[col_fecha], errors='coerce')
        fecha_min = col.min()
        fecha_max = col.max()
    return {
        'dataset':            nombre,
        'filas':              len(df),
        'columnas':           len(df.columns),
        'nulos_pct_max':      round(df.isnull().mean().max() * 100, 2),
        'duplicados_exactos': int(df.duplicated().sum()),
        'fecha_min':          fecha_min,
        'fecha_max':          fecha_max,
    }

In [4]:
# Columna de fecha ancla por entidad — nombres del contrato Bronze, estables
# entre restaurantes (no son específicos del original).
COL_FECHA_POR_ENTIDAD = {
    'tickets': 'date', 'ventas': 'report_start', 'reservas': 'reservation_datetime',
    'festivos': 'fecha', 'eventos': 'fecha_inicio', 'meteo_diaria': 'date',
    'meteo_horaria': 'datetime',
}

auditoria = pd.DataFrame([
    audit_dataset(nombre, df, col_fecha=COL_FECHA_POR_ENTIDAD.get(nombre))
    for nombre, df in datasets.items()
])
display(auditoria)

,dataset,filas,columnas,nulos_pct_max,duplicados_exactos,fecha_min,fecha_max
0,tickets,7386,12,100.0,0,2025-09-16 00:00:00,2026-09-13 00:00:00
1,ventas,2600,14,100.0,0,2025-09-15 00:00:00,2026-09-07 00:00:00
2,reservas,6500,21,100.0,0,2025-09-16 13:30:00,2026-10-14 22:58:00
3,articulos,50,7,100.0,0,NaT,NaT
4,total_articles,50,14,100.0,0,NaT,NaT
5,facturas,80,20,100.0,0,NaT,NaT


## Sección 2 — Funciones genéricas de depuración

Estas funciones son el motor de este notebook: implementan, de forma
reutilizable, los mismos patrones de decisión que el notebook original
aplicaba a mano dataset por dataset. Usan `atipicosAmissing`,
`analizar_variables_categoricas` y `patron_perdidos` de `tfm_depuracion.py`
como base estadística.

In [5]:
from tfm_depuracion import atipicosAmissing, analizar_variables_categoricas, patron_perdidos


def preguntar(mensaje: str, por_defecto: bool = False) -> bool:
    """Pregunta sí/no por consola. Enter = por_defecto."""
    sufijo = '[S/n]' if por_defecto else '[s/N]'
    resp = input(f'{mensaje} {sufijo}: ').strip().lower()
    if resp == '':
        return por_defecto
    return resp in ('s', 'si', 'sí', 'y', 'yes')


def resumen_basico(nombre: str, df: pd.DataFrame) -> None:
    print(f'=== {nombre.upper()} ===')
    print(f'Shape: {df.shape}')
    nulos = df.isnull().mean().mul(100).round(2)
    if (nulos > 0).any():
        print('\nNulos (%):')
        print(nulos[nulos > 0].sort_values(ascending=False).to_string())
    else:
        print('\nSin nulos.')
    print(f'\nDuplicados exactos: {df.duplicated().sum()}')


def sugerir_drop_columnas_por_nulos(df: pd.DataFrame, umbral: float = 70.0) -> pd.DataFrame:
    """Columnas con más de `umbral`% de nulos: se preguntan una a una antes
    de eliminarlas (nunca se eliminan solas)."""
    df = df.copy()
    nulos = df.isnull().mean().mul(100)
    candidatas = nulos[nulos > umbral].sort_values(ascending=False)
    if candidatas.empty:
        return df
    print(f'\nColumnas con más del {umbral:.0f}% de nulos:')
    for col, pct in candidatas.items():
        if preguntar(f"  '{col}' tiene {pct:.1f}% nulos. ¿Eliminarla?", por_defecto=True):
            df = df.drop(columns=[col])
            print(f'    -> eliminada.')
        else:
            print(f'    -> conservada.')
    return df


def flag_outliers_numericos(df: pd.DataFrame, excluir: list[str] | None = None,
                             umbral_zero_inflated: float = 0.60) -> pd.DataFrame:
    """Añade flag_outlier_<col> para cada columna numérica, vía
    atipicosAmissing (criterio doble MAD+IQR). Nunca borra ni modifica el
    valor original — solo añade una columna informativa.

    Las columnas con más de `umbral_zero_inflated` de ceros se excluyen
    automáticamente (el mismo motivo documentado para precipitación en el
    notebook original: con mediana y MAD ~0, el criterio degenera y marca
    como atípico cualquier valor moderado).
    """
    df = df.copy()
    excluir = set(excluir or [])
    numericas = [c for c in df.select_dtypes(include='number').columns if c not in excluir]
    for col in numericas:
        serie = df[col].dropna()
        if serie.empty:
            continue
        pct_ceros = (serie == 0).mean()
        if pct_ceros > umbral_zero_inflated:
            print(f"  '{col}': {pct_ceros*100:.0f}% ceros -> variable zero-inflated, "
                  f"se omite la detección de outliers (criterio no fiable aquí).")
            continue
        _, n_outliers = atipicosAmissing(serie)
        if n_outliers > 0:
            umbral_col = serie.quantile(0.999)
            df[f'flag_outlier_{col}'] = df[col] > umbral_col
            print(f"  '{col}': {n_outliers} outliers (criterio doble) -> "
                  f"flag_outlier_{col} añadida (umbral p99.9={umbral_col:.2f}).")
    return df


def detectar_dias_cierre(df: pd.DataFrame, col_fecha: str,
                          umbral_relativo: float = 0.10) -> pd.DataFrame:
    """Detecta si algún día de la semana tiene actividad casi nula frente a
    la media del resto, y pregunta si se marca como día de cierre."""
    df = df.copy()
    fechas = pd.to_datetime(df[col_fecha], errors='coerce').dt.date
    actividad = fechas.value_counts()
    dias_semana = pd.to_datetime(actividad.index).dayofweek
    por_dia = pd.Series(actividad.values, index=dias_semana).groupby(level=0).mean()
    media_general = por_dia.mean()
    nombres_dia = ['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo']

    df['dia_semana'] = pd.to_datetime(df[col_fecha], errors='coerce').dt.dayofweek
    df['es_dia_cierre'] = False
    for dia, media_dia in por_dia.items():
        if media_dia < media_general * umbral_relativo:
            nombre_dia = nombres_dia[dia]
            print(f"  Actividad en {nombre_dia}: {media_dia:.1f} registros/día "
                  f"vs media {media_general:.1f} -> posible día de cierre.")
            if preguntar(f"  ¿Marcar los {nombre_dia} como día de cierre "
                         f"(es_dia_cierre)?", por_defecto=True):
                df.loc[df['dia_semana'] == dia, 'es_dia_cierre'] = True
    return df


def normalizar_categorias_similares(df: pd.DataFrame, columna: str) -> pd.DataFrame:
    """Detecta valores de una categórica que son iguales salvo mayúsculas/
    guiones/espacios (p.ej. 'app-movil' vs 'appmovil') y pregunta si se
    fusionan bajo la forma más frecuente."""
    df = df.copy()
    if columna not in df.columns:
        return df
    valores = df[columna].dropna().unique()

    def _normalizar(v):
        return ''.join(ch for ch in str(v).lower() if ch.isalnum())

    grupos: dict[str, list] = {}
    for v in valores:
        grupos.setdefault(_normalizar(v), []).append(v)

    for _, variantes in grupos.items():
        if len(variantes) <= 1:
            continue
        conteos = df[columna].value_counts()
        canonico = max(variantes, key=lambda v: conteos.get(v, 0))
        otros = [v for v in variantes if v != canonico]
        print(f"  Posibles duplicados en '{columna}': {variantes} "
              f"-> candidata a forma única: '{canonico}'")
        if preguntar(f"  ¿Unificar {otros} bajo '{canonico}'?", por_defecto=True):
            df[columna] = df[columna].replace({v: canonico for v in otros})
            print(f"    -> unificado.")
    return df


def verificar_coherencia_suma(df: pd.DataFrame, col_a: str, col_b: str,
                               col_total: str, tol: float = 0.01) -> pd.DataFrame:
    """Comprueba que col_a + col_b == col_total (con tolerancia) y añade un
    flag. Nunca modifica valores, solo informa."""
    df = df.copy()
    faltan = [c for c in (col_a, col_b, col_total) if c not in df.columns]
    if faltan:
        return df
    suma = (df[col_a].fillna(0) + df[col_b].fillna(0)).round(2)
    diff = (suma - df[col_total].round(2)).abs()
    df['flag_incoherencia_suma'] = diff > tol
    n_incoherentes = df['flag_incoherencia_suma'].sum()
    print(f"  Coherencia {col_a} + {col_b} == {col_total}: "
          f"{n_incoherentes} filas incoherentes de {len(df)}.")
    return df


def verificar_integridad_referencial(hijo: pd.Series, padre: pd.Series, nombre: str) -> None:
    """Solo informa — nunca modifica nada. Decidir qué hacer con huérfanos
    referenciales depende demasiado del caso como para automatizarlo."""
    huerfanos = set(hijo.dropna().unique()) - set(padre.dropna().unique())
    if huerfanos:
        ejemplo = list(huerfanos)[:10]
        print(f"  {nombre}: {len(huerfanos)} valores sin correspondencia en el "
              f"maestro (ejemplo: {ejemplo})")
    else:
        print(f"  {nombre}: integridad referencial OK.")


def detectar_amount_units_incoherentes(df: pd.DataFrame, col_amount: str,
                                        col_units: str) -> pd.DataFrame:
    """Patrón amount<=0 con units>0 (habitual: invitaciones/cortesías, o
    descuadres de redondeo de IVA). Se flaguea siempre; corregir el importe
    a 0 se pregunta, porque modifica un valor."""
    df = df.copy()
    if col_amount not in df.columns or col_units not in df.columns:
        return df
    patron = (df[col_amount] <= 0) & (df[col_units] > 0)
    n = patron.sum()
    if n == 0:
        return df
    df['es_invitacion'] = patron
    print(f"  {n} filas con {col_amount}<=0 y {col_units}>0 -> flag es_invitacion añadida.")
    if preguntar(f"  ¿Corregir el importe de esas {n} filas a 0.0 (posibles "
                 f"invitaciones/cortesías o descuadres de redondeo)?", por_defecto=False):
        df.loc[patron, col_amount] = 0.0
        print(f"    -> importe corregido a 0.0 en {n} filas.")
    return df

## Sección 3 — Depuración por entidad

In [6]:
clean: dict[str, pd.DataFrame] = {}  # se va rellenando en cada subsección

### 3.1 Tickets

In [7]:
if 'tickets' in datasets:
    df = datasets['tickets'].copy()
    resumen_basico('tickets', df)

    for col in ['report_start', 'report_end', 'report_generated_on', 'date']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    print()
    df = flag_outliers_numericos(df, excluir=['receipt_count'])
    print()
    df = detectar_dias_cierre(df, col_fecha='date')

    clean['tickets'] = df
    print(f"\nFilas: {len(datasets['tickets'])} -> {len(df)} | "
          f"Columnas: {datasets['tickets'].shape[1]} -> {df.shape[1]}")
else:
    print('tickets no disponible, se salta esta sección.')

=== TICKETS ===
Shape: (7386, 12)

Nulos (%):
report_start           100.0
report_end             100.0
report_generated_on    100.0
terminal_start         100.0
terminal_end           100.0
turn                   100.0
receipt_count          100.0

Duplicados exactos: 0



Filas: 7386 -> 7386 | Columnas: 12 -> 14


### 3.2 Ventas semanales / total_articles\n\nMismo esquema y mismo tratamiento para ambas.

In [8]:
for nombre in ['ventas', 'total_articles']:
    if nombre not in datasets:
        print(f'{nombre} no disponible, se salta.')
        continue
    df = datasets[nombre].copy()
    resumen_basico(nombre, df)

    for col in ['report_start', 'report_end', 'report_generated_on']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    print()
    df = detectar_amount_units_incoherentes(df, col_amount='amount', col_units='units')
    print()
    df = flag_outliers_numericos(df, excluir=['department_code', 'article_code'])

    clean[nombre] = df
    print(f"\nFilas: {len(datasets[nombre])} -> {len(df)} | "
          f"Columnas: {datasets[nombre].shape[1]} -> {df.shape[1]}\n")

=== VENTAS ===
Shape: (2600, 14)

Nulos (%):
report_generated_on    100.0
terminal_start         100.0
terminal_end           100.0
turn                   100.0
department_code        100.0

Duplicados exactos: 0


  'units': 3 outliers (criterio doble) -> flag_outlier_units añadida (umbral p99.9=109.80).
  'amount': 83 outliers (criterio doble) -> flag_outlier_amount añadida (umbral p99.9=2181.77).

Filas: 2600 -> 2600 | Columnas: 14 -> 16

=== TOTAL_ARTICLES ===
Shape: (50, 14)

Nulos (%):
report_start           100.0
report_end             100.0
report_generated_on    100.0
terminal_start         100.0
terminal_end           100.0
turn                   100.0
department_code        100.0

Duplicados exactos: 0



Filas: 50 -> 50 | Columnas: 14 -> 14



### 3.3 Reservas

In [9]:
if 'reservas' in datasets:
    df = datasets['reservas'].copy()
    resumen_basico('reservas', df)

    print()
    df = sugerir_drop_columnas_por_nulos(df, umbral=70.0)

    print()
    df = normalizar_categorias_similares(df, 'origin')
    if 'status' in df.columns:
        print("\nValores únicos en 'status' (revisa si conviene agruparlos a mano "
              "más adelante, específico de cada TPV):")
        print(df['status'].value_counts(dropna=False).to_string())

    print()
    df = detectar_dias_cierre(df, col_fecha='reservation_datetime')

    if {'created_datetime', 'reservation_datetime'}.issubset(df.columns):
        walkin_like = (
            pd.to_datetime(df['created_datetime'], errors='coerce')
            > pd.to_datetime(df['reservation_datetime'], errors='coerce')
        )
        n_walkin = walkin_like.sum()
        if n_walkin > 0:
            print(f"\n{n_walkin} filas con created_datetime posterior a "
                  f"reservation_datetime (típico de walk-ins). Se flaguean, no se tocan.")
            df['flag_posible_walkin'] = walkin_like

    # Excluir reservas fuera del rango observable (más allá del último ticket)
    if 'tickets' in clean and 'date' in clean['tickets'].columns:
        fecha_limite = clean['tickets']['date'].max()
        futuras = pd.to_datetime(df['reservation_date'], errors='coerce') > fecha_limite
        n_futuras = futuras.sum()
        if n_futuras > 0:
            print(f"\n{n_futuras} reservas con fecha posterior al último ticket "
                  f"disponible ({fecha_limite.date()}).")
            if preguntar('¿Excluirlas del Silver por estar fuera del periodo '
                         'histórico analizable?', por_defecto=True):
                df = df[~futuras]
                print(f'  -> excluidas.')

    clean['reservas'] = df
    print(f"\nFilas: {len(datasets['reservas'])} -> {len(df)} | "
          f"Columnas: {datasets['reservas'].shape[1]} -> {df.shape[1]}")
else:
    print('reservas no disponible, se salta esta sección.')

=== RESERVAS ===
Shape: (6500, 21)

Nulos (%):
referrer            100.00
created_date        100.00
created_time        100.00
restaurant          100.00
reservation_type    100.00
entered_by          100.00
group               100.00
table                 2.31

Duplicados exactos: 0


Columnas con más del 70% de nulos:
    -> eliminada.
    -> conservada.
    -> conservada.
    -> eliminada.
    -> eliminada.
    -> eliminada.
    -> conservada.


Valores únicos en 'status' (revisa si conviene agruparlos a mano más adelante, específico de cada TPV):
status
Completada                  4988
Cancelada por el cliente     748
Liberada                     449
No show                      165
Confirmada                   150


150 reservas con fecha posterior al último ticket disponible (2026-09-13).

Filas: 6500 -> 6500 | Columnas: 21 -> 19


### 3.4 Tips (propinas)

In [10]:
if 'tips' in datasets:
    df = datasets['tips'].copy()
    resumen_basico('tips', df)

    print()
    df = verificar_coherencia_suma(df, 'document_amount', 'tip', 'document_total')

    if {'tip', 'document_amount'}.issubset(df.columns):
        anomalas = df['tip'] > df['document_amount']
        if anomalas.sum() > 0:
            df['flag_tip_anomala'] = anomalas
            print(f"\n{anomalas.sum()} propinas superiores al importe del pedido "
                  f"-> flag_tip_anomala añadida.")

    print()
    df = flag_outliers_numericos(df)

    clean['tips'] = df
    print(f"\nFilas: {len(datasets['tips'])} -> {len(df)} | "
          f"Columnas: {datasets['tips'].shape[1]} -> {df.shape[1]}")
else:
    print('tips no disponible, se salta esta sección.')

tips no disponible, se salta esta sección.


### 3.5 Catálogos — Artículos, Departamentos y Menú

In [11]:
for nombre in ['articulos', 'departamentos', 'menu']:
    if nombre not in datasets:
        print(f'{nombre} no disponible, se salta.')
        continue
    df = datasets[nombre].copy()
    resumen_basico(nombre, df)
    print()
    df = sugerir_drop_columnas_por_nulos(df, umbral=70.0)
    clean[nombre] = df
    print(f"\nFilas: {len(datasets[nombre])} -> {len(df)} | "
          f"Columnas: {datasets[nombre].shape[1]} -> {df.shape[1]}\n")

# ── Validación referencial entre catálogos y transaccionales (solo informa) ──
if 'articulos' in clean and 'departamentos' in clean:
    verificar_integridad_referencial(
        clean['articulos']['department_code'], clean['departamentos']['department_code'],
        'department_code en articulos vs departamentos',
    )
if 'ventas' in clean and 'articulos' in clean:
    verificar_integridad_referencial(
        clean['ventas']['article_code'], clean['articulos']['article_code'],
        'article_code en ventas vs articulos',
    )

=== ARTICULOS ===
Shape: (50, 7)

Nulos (%):
article_short_name    100.0
department_code       100.0

Duplicados exactos: 0


Columnas con más del 70% de nulos:
    -> eliminada.
    -> eliminada.

Filas: 50 -> 50 | Columnas: 7 -> 5

departamentos no disponible, se salta.
menu no disponible, se salta.
  article_code en ventas vs articulos: integridad referencial OK.


### 3.6 Meteorología

In [12]:
if 'meteo_diaria' in datasets:
    df = datasets['meteo_diaria'].copy()
    resumen_basico('meteo_diaria', df)

    # Renombrado genérico: quitar unidades embebidas entre paréntesis del
    # nombre de columna (incompatibles con la mayoría de librerías de ML),
    # conservando la unidad en un comentario, no en el propio nombre.
    import re
    renombres = {}
    for col in df.columns:
        m = re.match(r'^(.*?)\s*\(([^)]+)\)\s*$', col)
        if m:
            base, unidad = m.group(1).strip(), m.group(2).strip()
            nuevo = re.sub(r'[^a-zA-Z0-9_]+', '_', base).strip('_').lower()
            renombres[col] = nuevo
    if renombres:
        print('\nRenombrado de columnas (unidades fuera del nombre):')
        for antes, despues in renombres.items():
            print(f'  {antes!r} -> {despues!r}')
        df = df.rename(columns=renombres)

    # 'date' llega como texto plano (así lo guarda el propio Bronze del
    # proyecto) -> se convierte a datetime para poder cruzarla con otros
    # datasets (tickets, festivos) en la Sección 4.
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')

    print()
    df = flag_outliers_numericos(df)

    clean['meteo_diaria'] = df
    print(f"\nFilas: {len(datasets['meteo_diaria'])} -> {len(df)} | "
          f"Columnas: {datasets['meteo_diaria'].shape[1]} -> {df.shape[1]}")
else:
    print('meteo_diaria no disponible, se salta esta sección.')

print()
if 'meteo_horaria' in datasets:
    clean['meteo_horaria'] = datasets['meteo_horaria'].copy()
    print(f"meteo_horaria: sin cambios ({clean['meteo_horaria'].shape}), se "
          f"preserva en Silver aunque no se use en el modelo base.")
else:
    print('meteo_horaria no disponible, se salta esta sección.')

meteo_diaria no disponible, se salta esta sección.

meteo_horaria no disponible, se salta esta sección.


### 3.7 Festivos y eventos\n\n`total_articles` ya se depuró junto a `ventas` en 3.2, al compartir esquema.

In [13]:
if 'festivos' in datasets:
    df = datasets['festivos'].copy()
    resumen_basico('festivos', df)
    if 'fecha' in df.columns:
        df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce', dayfirst=True)
    clean['festivos'] = df
    print(f"Rango: {df['fecha'].min()} -> {df['fecha'].max()}" if 'fecha' in df.columns else '')
else:
    print('festivos no disponible, se salta.')

print()
if 'eventos' in datasets:
    df = datasets['eventos'].copy()
    resumen_basico('eventos', df)
    for col in ['fecha_inicio', 'fecha_fin']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
    if {'fecha_inicio', 'fecha_fin'}.issubset(df.columns):
        incoherentes = df['fecha_inicio'] > df['fecha_fin']
        if incoherentes.sum() > 0:
            print(f"\n{incoherentes.sum()} eventos con fecha_inicio posterior a fecha_fin.")
            df['flag_fechas_evento_incoherentes'] = incoherentes

    # No hay forma de saber, de forma genérica, qué categorías de evento
    # importan para ESTE restaurante (dependía de conocimiento experto del
    # original: proximidad, tipo de público, etc.). Se pregunta en vez de
    # inventar un criterio.
    if 'categoria' in df.columns:
        conteo_cat = df['categoria'].value_counts()
        print('\nCategorías de evento encontradas:')
        print(conteo_cat.to_string())
        respuesta = input(
            '\n¿Excluir alguna categoría del Silver? Escribe los nombres '
            'separados por coma, o pulsa Enter para no excluir ninguna: '
        ).strip()
        if respuesta:
            excluir = [c.strip() for c in respuesta.split(',')]
            antes = len(df)
            df = df[~df['categoria'].isin(excluir)]
            print(f'  -> excluidas {antes - len(df)} filas de categorías {excluir}.')

    clean['eventos'] = df
    print(f"\nFilas: {len(datasets['eventos'])} -> {len(df)} | "
          f"Columnas: {datasets['eventos'].shape[1]} -> {df.shape[1]}")
else:
    print('eventos no disponible, se salta.')

festivos no disponible, se salta.

eventos no disponible, se salta.


### 3.8 Facturas (PDF)

In [14]:
if 'facturas' in datasets:
    df = datasets['facturas'].copy()
    resumen_basico('facturas', df)

    if 'fecha' in df.columns:
        df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce', dayfirst=True)
    if 'hora' in df.columns:
        df['hora'] = df['hora'].astype('string').str.strip()

    print()
    if {'total', 'suma_items'}.issubset(df.columns) and 'cuadra_total' not in df.columns:
        df = verificar_coherencia_suma(df, 'suma_items', 'diferencia_total_vs_items', 'total')
    elif 'cuadra_total' in df.columns:
        col_bool = df['cuadra_total'].astype('boolean')
        n_no_cuadra = (col_bool == False).sum()
        n_desconocido = col_bool.isna().sum()
        if n_desconocido == len(df):
            print("  'cuadra_total' viene vacía en todas las filas: este origen no trae "
                  "líneas de producto de las que derivar esa validación (no es un fallo, "
                  "es una limitación real de la fuente). No se puede evaluar el cuadre.")
        else:
            print(f"  Facturas donde no cuadra el total: {n_no_cuadra} de {len(df)} "
                  f"({n_desconocido} sin dato suficiente para saberlo).")

    if 'tarjeta' in df.columns:
        df['pago_tarjeta'] = df['tarjeta'].notna()
        print(f"  Pagos con tarjeta: {df['pago_tarjeta'].sum()} de {len(df)}.")

    print()
    df = sugerir_drop_columnas_por_nulos(df, umbral=70.0)
    print()
    df = flag_outliers_numericos(df, excluir=['num_items'])

    # porcentaje_iva constante no aporta como feature -> se pregunta, no se
    # asume (podría no ser constante para otro restaurante con varios tipos de IVA)
    if 'porcentaje_iva' in df.columns and df['porcentaje_iva'].nunique() == 1:
        valor = df['porcentaje_iva'].iloc[0]
        print(f"\n'porcentaje_iva' es constante ({valor}) en todas las filas.")
        if preguntar("¿Eliminarla por no aportar como feature?", por_defecto=True):
            df = df.drop(columns=['porcentaje_iva'])

    clean['facturas'] = df
    print(f"\nFilas: {len(datasets['facturas'])} -> {len(df)} | "
          f"Columnas: {datasets['facturas'].shape[1]} -> {df.shape[1]}")
else:
    print('facturas no disponible, se salta esta sección.')

=== FACTURAS ===
Shape: (80, 20)

Nulos (%):
archivo_pdf                  100.0
ruta_pdf                     100.0
restaurante                  100.0
cif                          100.0
telefono                     100.0
porcentaje_iva               100.0
suma_items                   100.0
diferencia_total_vs_items    100.0
cuadra_total                 100.0

Duplicados exactos: 0

  'cuadra_total' viene vacía en todas las filas: este origen no trae líneas de producto de las que derivar esa validación (no es un fallo, es una limitación real de la fuente). No se puede evaluar el cuadre.
  Pagos con tarjeta: 80 de 80.


Columnas con más del 70% de nulos:
    -> eliminada.
    -> eliminada.
    -> eliminada.
    -> eliminada.
    -> eliminada.
    -> conservada.
    -> conservada.
    -> conservada.
    -> conservada.

  'efectivo': 65% ceros -> variable zero-inflated, se omite la detección de outliers (criterio no fiable aquí).

Filas: 80 -> 80 | Columnas: 20 -> 16


## Sección 4 — Validación cruzada entre datasets

In [15]:
# ── 1. Cobertura temporal ─────────────────────────────────────────────────
coberturas = {}
for nombre, col in [('tickets', 'date'), ('ventas', 'report_start'),
                     ('reservas', 'reservation_datetime'), ('meteo_diaria', 'date'),
                     ('festivos', 'fecha'), ('eventos', 'fecha_inicio')]:
    if nombre in clean and col in clean[nombre].columns:
        serie = pd.to_datetime(clean[nombre][col], errors='coerce')
        coberturas[nombre] = (serie.min(), serie.max())

if coberturas:
    df_cob = pd.DataFrame(coberturas, index=['fecha_min', 'fecha_max']).T
    print('Cobertura temporal por dataset:')
    display(df_cob)

Cobertura temporal por dataset:


,fecha_min,fecha_max
tickets,2025-09-16 00:00:00,2026-09-13 00:00:00
ventas,2025-09-15 00:00:00,2026-09-07 00:00:00
reservas,2025-09-16 13:30:00,2026-10-14 22:58:00


In [16]:
# ── 2. ¿Todos los días con tickets tienen meteorología? ─────────────────────
if 'tickets' in clean and 'meteo_diaria' in clean and 'date' in clean['meteo_diaria'].columns:
    merge_meteo = (
        clean['tickets'][['date']].drop_duplicates()
        .merge(clean['meteo_diaria'][['date']], on='date', how='left', indicator=True)
    )
    sin_meteo = (merge_meteo['_merge'] == 'left_only').sum()
    print(f'Días con tickets sin datos meteorológicos: {sin_meteo}')
    print('Cobertura meteorológica completa' if sin_meteo == 0 else 'REVISAR: hay días sin meteo')
else:
    print('No se puede comprobar cobertura meteo x tickets (falta algún dataset).')

No se puede comprobar cobertura meteo x tickets (falta algún dataset).


In [17]:
# ── 3. Cobertura de festivos sobre días con tickets ──────────────────────────
if 'tickets' in clean and 'festivos' in clean and 'fecha' in clean['festivos'].columns:
    fechas_festivas = set(clean['festivos']['fecha'].dt.date)
    dias_tickets = clean['tickets'][['date']].drop_duplicates().copy()
    dias_tickets['date_only'] = dias_tickets['date'].dt.date
    dias_tickets['es_festivo'] = dias_tickets['date_only'].isin(fechas_festivas).astype(int)
    print(f"Días con tickets: {len(dias_tickets)} | de ellos festivos: "
          f"{dias_tickets['es_festivo'].sum()}")
else:
    print('No se puede comprobar cobertura de festivos (falta algún dataset).')

No se puede comprobar cobertura de festivos (falta algún dataset).


## Sección 5 — Guardado Silver

In [18]:
def preparar_para_parquet(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte columnas object con tipos mixtos a string puro antes de
    guardar (igual que en 02_original)."""
    df = df.copy()
    for col in df.select_dtypes(include='object').columns:
        if df[col].apply(type).nunique() > 1:
            df[col] = df[col].astype(str)
    return df

In [19]:
for nombre, df in clean.items():
    ruta = SILVER_SNAP / f'{nombre}_silver.parquet'
    preparar_para_parquet(df).to_parquet(ruta, index=False)

print(f'\n✓ {len(clean)}/{len(NOMBRES_ENTIDAD)} datasets guardados en {SILVER_SNAP}')
if len(clean) < len(NOMBRES_ENTIDAD):
    faltan_silver = set(NOMBRES_ENTIDAD) - set(clean.keys())
    print(f'  No se generaron (no estaban en Bronze, o se saltaron): {sorted(faltan_silver)}')
print(f'  Total filas Silver: {sum(len(df) for df in clean.values()):,}')


✓ 6/13 datasets guardados en /Users/laura/TFM-Hosteleria-AI/agente/prueba3_silver
  No se generaron (no estaban en Bronze, o se saltaron): ['departamentos', 'eventos', 'festivos', 'menu', 'meteo_diaria', 'meteo_horaria', 'tips']
  Total filas Silver: 16,666
